# Démo exports spatiaux avec le test set GPS

Ce notebook montre comment préparer des tables spatiales interrogeables à partir du test set local :

1. charger les tables landed du test set ;
2. transformer les traces avec `xyt_gps` ;
3. convertir les legs en points indexés H3 ;
4. agréger les fréquentations par cellule ;
5. exporter en GeoParquet et DuckDB ;
6. interroger les résultats avec SQL.

Les fichiers écrits vont dans `Data/Output/test-set-results/spatial-analytics`, un dossier ignoré par git.

## 1. Préparer l'environnement

La cellule suivante retrouve le package local et l'importe depuis `src`, afin d'utiliser directement le code en cours de développement.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd


def find_package_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for base in (start, *start.parents):
        for candidate in (base, base / "package-xyt-gps"):
            if (candidate / "src" / "xyt_gps").exists():
                return candidate.resolve()
    raise FileNotFoundError("Impossible de trouver package-xyt-gps/src/xyt_gps.")


package_root = find_package_root()
import xyt_gps as xyt
pd.set_option("display.max_columns", 80)
print(f"xyt_gps chargé depuis : {xyt.__file__}")
package_root

## 1 bis. Vérifier les dépendances optionnelles

Cette démo utilise les extras `analytics`, `export` et `viz` : DuckDB, H3, PyArrow et Folium. Si la cellule suivante échoue, elle affiche une commande `%pip install` avec les chemins exacts à lancer dans le kernel actif.

In [ ]:
required_modules = {
    "duckdb": "requirements-analytics.txt",
    "h3": "requirements-analytics.txt",
    "pyarrow": "requirements-export.txt",
    "folium": "requirements-viz.txt",
}

install_command = (
    "%pip install "
    f"-r {package_root / 'requirements-analytics.txt'} "
    f"-r {package_root / 'requirements-export.txt'} "
    f"-r {package_root / 'requirements-viz.txt'}"
)

missing = []
for module_name, requirement_file in required_modules.items():
    try:
        __import__(module_name)
    except ImportError:
        missing.append((module_name, requirement_file))

if missing:
    missing_text = ", ".join(f"{module} ({req})" for module, req in missing)
    raise ImportError(
        "Dépendances optionnelles manquantes dans ce kernel : "
        f"{missing_text}. Lancez cette commande dans une cellule du notebook, "
        f"puis redémarrez le kernel :\n{install_command}"
    )

"Toutes les dépendances optionnelles de la démo sont disponibles."

## 2. Retrouver ou générer le test set

Le test set attendu est `Data/Output/test-set-gps`. S'il n'existe pas, le générateur local le reconstruit à partir de la source anonymisée et spatialement altérée disponible dans le dépôt de travail.

In [ ]:
test_set_dir = (package_root.parent / "Data" / "Output" / "test-set-gps").resolve()
if not test_set_dir.exists():
    generator = package_root / "examples" / "build_generic_test_set.py"
    subprocess.run([sys.executable, str(generator)], check=True)

manifest_path = test_set_dir / "test_set_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
print(test_set_dir)
manifest.get("tables", {})

## 3. Charger les tables landed

On charge les tables du test set sans passer par une fonction d'import du package. À ce stade, l'objectif est simplement de disposer des DataFrames attendus.

In [ ]:
storyline = pd.read_parquet(test_set_dir / "storyline.parquet")
trips = pd.read_parquet(test_set_dir / "trips.parquet")
journeys = pd.read_parquet(test_set_dir / "journeys.parquet")
user_statistics = pd.read_csv(test_set_dir / "user_statistics.csv", low_memory=False)
user_presence = pd.read_parquet(test_set_dir / "user_presence.parquet")

input_summary = pd.DataFrame(
    [
        {"table": "storyline", "rows": len(storyline), "users": storyline["user_id"].nunique()},
        {"table": "trips", "rows": len(trips), "users": trips["user_id"].nunique()},
        {"table": "journeys", "rows": len(journeys), "users": journeys["user_id"].nunique()},
        {"table": "user_statistics", "rows": len(user_statistics), "users": user_statistics["user_id"].nunique() if "user_id" in user_statistics else 0},
        {"table": "user_presence", "rows": len(user_presence), "users": user_presence["user_id"].nunique()},
    ]
)
input_summary

## 4. Valider le schéma minimal

La validation sert à vérifier que les colonnes nécessaires au chargement existent avant de lancer la transformation.

In [ ]:
schema_report = xyt.check_raw_import_columns(
    storyline,
    user_statistics,
    trips=trips,
    journeys=journeys,
    raise_on_error=True,
)
schema_report

## 5. Construire la configuration de transformation

Le test set contient une expérimentation générique avec trois phases. Dans un autre projet, cette configuration peut être minimale : `xyt.ProjectConfig()` suffit si l'analyse ne dépend pas de phases.

In [ ]:
landing_config = json.loads((test_set_dir / "landing_config.json").read_text(encoding="utf-8"))
experiment = landing_config["experiments"][0]

phases = []
for phase_index in (1, 2, 3):
    start, end = experiment.get(f"phase_{phase_index}", (None, None))
    if start and end:
        phases.append(xyt.Phase(f"Phase{phase_index}", start, end))

config = xyt.ProjectConfig(
    experiment_name=experiment.get("experiment_name", "test_set"),
    motiontag_project_name=experiment.get("motiontag_project_name", "test_set"),
    period=experiment.get("dump_date_range", "test_set"),
    raw_data_dir=test_set_dir,
    phases=tuple(phases),
    mappings=xyt.mode_purpose_mapping(),
    time_slices=(
        xyt.TimeSlice("HPM", "07:10", "09:00"),
        xyt.TimeSlice("HPS", "17:30", "20:00"),
    ),
)
config

## 6. Transformer les données avec le package

On produit ici les tables structurées : `legs`, `staypoints`, `trips`, `journeys`, `user_stats` et tables de correspondance.

In [ ]:
raw = xyt.RawGpsData(
    storyline=storyline,
    trips=trips,
    journeys=journeys,
    user_statistics=user_statistics,
)

dataset = xyt.prepare_mobility_dataset(
    raw,
    config,
    resample_missing_days=True,
    clean_leg_geometries=True,
    add_length_outlier_flags=True,
    add_signal_quality_flags=True,
)

pd.DataFrame(
    [
        {"table": table_name, "rows": len(table), "users": table["user_id"].nunique() if "user_id" in table.columns else None}
        for table_name, table in xyt.mobility_dataset_tables(dataset).items()
    ]
)

## 7. Créer les tables H3

`build_spatial_analytics_tables()` crée trois tables inspectables :

- `leg_points_h3` : points de legs indexés H3 ;
- `h3_frequency` : table longue de fréquentation par cellule et dimensions ;
- `h3_count_matrix` : table large avec plusieurs colonnes de counts pour dashboard.

La démo utilise deux résolutions H3 (`8` et `9`) et ajoute une tranche horaire (`HPM`, `HC`, `HPS`) depuis `config.time_slices`.

In [ ]:
H3_RESOLUTIONS = [8, 9]
H3_MAP_RESOLUTION = 9
SAMPLE_DISTANCE_M = None  # Exemple : 50 pour échantillonner les lignes tous les 50 m.
COUNT_DIMENSION_SETS = (
    ("mode_niv1",),
    ("purpose_mrmt",),
    ("time_slice",),
    ("mode_niv1", "time_slice"),
    ("purpose_mrmt", "time_slice"),
)

spatial_tables = xyt.build_spatial_analytics_tables(
    dataset,
    h3_resolution=H3_RESOLUTIONS,
    config=config,
    frequency_group_cols=["mode_niv1", "purpose_mrmt", "time_slice", "phase"],
    count_dimension_sets=COUNT_DIMENSION_SETS,
    count_metrics=("point_count", "trip_count"),
    sample_distance_m=SAMPLE_DISTANCE_M,
)

leg_points_h3 = spatial_tables["leg_points_h3"]
h3_frequency = spatial_tables["h3_frequency"]
h3_count_matrix = spatial_tables["h3_count_matrix"]

leg_points_h3[[
    "h3_resolution",
    "leg_id",
    "user_id",
    "mode_niv1",
    "purpose_mrmt",
    "time_slice",
    "phase",
    "point_sequence",
    "h3_cell",
    "lon",
    "lat",
]].head()

## 8. Table longue de fréquentation

`h3_frequency` garde une ligne par cellule H3 et combinaison de dimensions : résolution, mode, motif, tranche horaire et phase. Elle sert aux analyses groupées et aux requêtes SQL.

In [ ]:
h3_frequency.head(20)

## 8 bis. Table large de counts pour dashboard

`h3_count_matrix` contient une ligne par cellule H3 et plusieurs colonnes de counts : total, par mode, par motif, par tranche horaire, puis croisements mode/tranche et motif/tranche.

In [ ]:
h3_count_matrix.filter(
    regex="^h3_|^point_count__total$|^trip_count__total$|mode_niv1|purpose_mrmt|time_slice"
).head()

In [ ]:
h3_summary = pd.DataFrame(
    [
        {"metric": "résolutions H3", "value": ", ".join(map(str, sorted(leg_points_h3["h3_resolution"].unique())))},
        {"metric": "points H3", "value": len(leg_points_h3)},
        {"metric": "cellules H3 uniques", "value": leg_points_h3["h3_cell"].nunique()},
        {"metric": "utilisateurs", "value": leg_points_h3["user_id"].nunique()},
        {"metric": "legs", "value": leg_points_h3["leg_id"].nunique()},
        {"metric": "colonnes h3_count_matrix", "value": h3_count_matrix.shape[1]},
    ]
)
h3_summary

## 9. Visualiser la fréquentation H3

La carte affiche les cellules H3 les plus fréquentées. Pour garder la carte réactive dans un navigateur, on limite ici l'affichage aux `max_cells` cellules les plus fréquentées.

In [ ]:
output_dir = (package_root.parent / "Data" / "Output" / "test-set-results" / "spatial-analytics").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

h3_map_data = (
    h3_count_matrix.loc[h3_count_matrix["h3_resolution"].eq(H3_MAP_RESOLUTION)]
    .rename(
        columns={
            "point_count__total": "point_count",
            "trip_count__total": "trip_count",
        }
    )
    [["h3_cell", "point_count", "trip_count"]]
)

h3_map = xyt.plot_h3_frequency_map(
    h3_map_data,
    value_col="point_count",
    max_cells=2000,
    save_path=output_dir / "h3_frequency_map.html",
)

h3_map

## 10. Exporter GeoParquet et DuckDB

Les tables déjà construites sont écrites dans `Data/Output/test-set-results/spatial-analytics`. Le notebook évite ainsi de recalculer les H3 au moment de l'export.

In [ ]:
output_dir = (package_root.parent / "Data" / "Output" / "test-set-results" / "spatial-analytics").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

spatial_manifest = xyt.write_spatial_analytics_tables(
    spatial_tables,
    output_dir,
    formats=("parquet", "csv"),
)

spatial_manifest

In [ ]:
duckdb_manifest = xyt.write_duckdb_spatial_database(
    {
        "leg_points_h3": leg_points_h3,
        "h3_frequency": h3_frequency,
        "h3_count_matrix": h3_count_matrix,
    },
    output_dir / "test_set_spatial.duckdb",
    load_spatial=True,
    require_spatial_extension=False,
)

duckdb_manifest

## 10. Interroger la base DuckDB

DuckDB permet de lire les tables en SQL. Même sans extension spatial, les colonnes `h3_cell`, `lon`, `lat`, modes et phases suffisent pour alimenter un dashboard.

In [ ]:
import duckdb

con = duckdb.connect(str(output_dir / "test_set_spatial.duckdb"))

query_top_cells = con.execute("""
    SELECT
        h3_resolution,
        h3_cell,
        mode_niv1,
        purpose_mrmt,
        time_slice,
        phase,
        point_count,
        trip_count,
        user_count,
        leg_count
    FROM h3_frequency
    WHERE h3_resolution = ?
    ORDER BY point_count DESC
    LIMIT 15
""", [H3_MAP_RESOLUTION]).fetchdf()

query_top_cells

In [ ]:
query_by_mode_time = con.execute("""
    SELECT
        mode_niv1,
        time_slice,
        COUNT(DISTINCT h3_cell) AS h3_cells,
        SUM(point_count) AS points,
        SUM(trip_count) AS trips,
        MAX(user_count) AS max_users_in_cell
    FROM h3_frequency
    WHERE h3_resolution = ?
    GROUP BY mode_niv1, time_slice
    ORDER BY points DESC
""", [H3_MAP_RESOLUTION]).fetchdf()

con.close()
query_by_mode_time

## À retenir

Pour une plateforme de visualisation, le flux recommandé est :

```text
MobilityDataset
  -> GeoParquet pour les tables géographiques durables
  -> H3 pour les cartes de fréquentation
  -> DuckDB pour requêter localement en SQL
  -> PostGIS ou PMTiles plus tard si le dashboard doit être servi en production
```

Le niveau `2-transformed-data` suffit pour une carte de fréquentation simple. Le niveau `3-enriched-data` est préférable si les cartes doivent intégrer les indicateurs, CO2, santé, qualité GPS ou filtres analytiques.